# Alternative Architecture: Neuromorphic Spiking Neural Networks

**Ethan Vu, Marcus Vu** — CMPE 257

This notebook carries out the full project proposal from start to finish: the problem statement, the CIFAR-10 pipeline, a CNN (ANN) baseline, a Spiking Neural Network trained with **Temporal Effective Batch Normalization (TEBN)**, and a side-by-side comparison of accuracy, training time, inference speed, GPU resource usage, and power/energy.

**Contents**

1. Problem Description
2. Dataset & Preprocessing
3. Setup & Configuration
4. Measurement Utilities (timing, GPU memory, power)
5. ANN Baseline (CNN)
6. SNN: LIF neurons, surrogate gradients, TEBN
7. Training & Evaluation: ANN
8. Training & Evaluation: SNN (TEBN)
9. Spike-Activity Analysis
10. Theoretical Energy Estimate (MACs vs. SOPs)
11. ANN vs. SNN Comparison
12. Optional Experiment: TEBN vs. Standard BN (ablation)
13. Expected Challenges
14. Team Roles
15. Conclusions

## 1. Problem Description

The modern paradigm of Artificial Neural Networks (ANNs) often requires massive amounts of power and water. As compute scales proportionally with a model's intelligence, Spiking Neural Networks (SNNs) offer a possible alternative for achieving continual increases in intelligence with less power.

Implementing *"Temporal Effective Batch Normalization in Spiking Neural Networks"* by Chaoteng Duan, Jianhao Ding, Shiyan Chen, Zhaofei Yu, and Tiejun Huang (NeurIPS 2022), we will train and compare the performance and energy consumption of ANN and SNN for image classification to determine whether or not SNNs provide any benefits.

This should have benefits for large AI corporations that are investing in data centers, as it provides a cheaper alternative architecture to scale large intelligent systems. It would allow less effects on the environment, improving communities that reside near these establishments.

**Reference paper:** [Temporal Effective Batch Normalization in Spiking Neural Networks (NeurIPS 2022)](https://proceedings.neurips.cc/paper_files/paper/2022/hash/de2ad3ed44ee4e675b3be42aa0b615d0-Abstract-Conference.html)

## 2. Dataset & Preprocessing

**CIFAR-10:** <https://www.cs.toronto.edu/~kriz/cifar.html>

CIFAR-10 is a commonly used image-classification dataset containing **60,000 RGB 32×32 images** in **10 classes** (50,000 train / 10,000 test).

**Preprocessing plan**

- Convert images into PyTorch tensors, then normalize with the per-channel CIFAR-10 mean and standard deviation.
- Training augmentation: random crop with padding, horizontal flip, color alteration (`ColorJitter`), and small rotations. The paper relies on augmentation to reach its reported accuracy.
- Since CIFAR-10 is already balanced and well established, no class-imbalance handling is needed.

Both models use **the same data pipeline** so the comparison is fair.

## 3. Setup & Configuration

The target hardware is a **GTX 1080 Ti** (11 GB). The notebook also runs on Google Colab or Kaggle if local training is too slow (see *Expected Challenges*).

- `QUICK_RUN = True` trains on a small subset for a couple of epochs, to check that everything runs end to end.
- `pynvml` (package `nvidia-ml-py`) reads GPU power draw. Without it, the notebook still runs but skips the measured-power numbers.

In [ ]:
# Uncomment on a fresh environment (Colab/Kaggle already have torch/torchvision):
# %pip install torch torchvision nvidia-ml-py matplotlib pandas

In [ ]:
import os, time, math, random, threading, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset

try:
    import pynvml
    pynvml.nvmlInit()
    HAS_NVML = True
except Exception:
    HAS_NVML = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU",
      "| power monitoring:", HAS_NVML)

In [ ]:
CFG = dict(
    QUICK_RUN     = False,   # True -> small subset, 2 epochs (sanity check)
    DATA_DIR      = "./data",
    BATCH_SIZE    = 128,
    NUM_WORKERS   = 4,       # set to 0 if DataLoader workers cause errors on Windows
    EPOCHS        = 30,      # the paper trains for 300; 30 is a practical budget on a 1080 Ti
    LR            = 0.1,
    MOMENTUM      = 0.9,
    WEIGHT_DECAY  = 5e-4,
    SEED          = 42,
    # SNN-specific (values follow the TEBN paper / official code)
    TIMESTEPS     = 4,       # T
    TAU           = 0.5,     # membrane decay factor
    V_TH          = 1.0,     # firing threshold
    SG_GAMMA      = 1.0,     # width of the triangle surrogate gradient
    # Energy constants (45 nm CMOS, Horowitz 2014), used by most SNN papers
    E_MAC_PJ      = 4.6,
    E_AC_PJ       = 0.9,
    RUN_ABLATION  = False,   # Section 12: also train an SNN with plain BN
)
if CFG["QUICK_RUN"]:
    CFG["EPOCHS"] = 2

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(CFG["SEED"])
CFG

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)
CLASSES = ("plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")

train_tf = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.RandomApply([T.ColorJitter(0.2, 0.2, 0.2, 0.05)], p=0.5),   # color alteration
    T.RandomRotation(10),                                          # rotation
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
])
test_tf = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])

train_set = torchvision.datasets.CIFAR10(CFG["DATA_DIR"], train=True,  download=True, transform=train_tf)
test_set  = torchvision.datasets.CIFAR10(CFG["DATA_DIR"], train=False, download=True, transform=test_tf)

if CFG["QUICK_RUN"]:
    train_set = Subset(train_set, range(2000))
    test_set  = Subset(test_set,  range(1000))

pin = DEVICE.type == "cuda"
train_loader = DataLoader(train_set, batch_size=CFG["BATCH_SIZE"], shuffle=True,
                          num_workers=CFG["NUM_WORKERS"], pin_memory=pin, drop_last=True,
                          persistent_workers=CFG["NUM_WORKERS"] > 0)
test_loader  = DataLoader(test_set, batch_size=256, shuffle=False,
                          num_workers=CFG["NUM_WORKERS"], pin_memory=pin,
                          persistent_workers=CFG["NUM_WORKERS"] > 0)
print(f"train: {len(train_set)} images | test: {len(test_set)} images")

In [ ]:
# Look at a few augmented training images
raw = torchvision.datasets.CIFAR10(CFG["DATA_DIR"], train=True, download=False, transform=train_tf)
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
mean, std = torch.tensor(CIFAR_MEAN)[:, None, None], torch.tensor(CIFAR_STD)[:, None, None]
for ax, i in zip(axes.flat, range(16)):
    img, y = raw[i]
    ax.imshow((img * std + mean).clamp(0, 1).permute(1, 2, 0)); ax.set_title(CLASSES[y]); ax.axis("off")
plt.suptitle("Augmented CIFAR-10 samples"); plt.tight_layout(); plt.show()

## 4. Measurement Utilities

The proposal calls for these metrics for **both** models:

| Metric | How it is measured |
|---|---|
| Accuracy | Top-1 on the 10,000-image test set |
| Training time | Wall-clock seconds per epoch and in total (CUDA-synchronized) |
| Inference speed | Throughput (images/s) over the test set, plus batch-1 latency |
| GPU resource usage | Peak allocated memory (`torch.cuda.max_memory_allocated`), mean GPU utilization (NVML) |
| Power consumption | NVML samples GPU board power every 100 ms in a background thread. Energy (J) = integral of power over time |

A second, hardware-independent energy estimate (MACs vs. synaptic operations) appears in Section 10.

In [ ]:
class GPUMonitor:
    # Samples GPU power (W), utilization (%) and memory (MB) in a background thread.
    def __init__(self, interval=0.1, index=0):
        self.interval, self.samples, self._stop = interval, [], threading.Event()
        self.handle = pynvml.nvmlDeviceGetHandleByIndex(index) if HAS_NVML else None

    def _run(self):
        while not self._stop.is_set():
            t = time.perf_counter()
            p = pynvml.nvmlDeviceGetPowerUsage(self.handle) / 1000.0
            u = pynvml.nvmlDeviceGetUtilizationRates(self.handle).gpu
            m = pynvml.nvmlDeviceGetMemoryInfo(self.handle).used / 2**20
            self.samples.append((t, p, u, m))
            time.sleep(self.interval)

    def __enter__(self):
        if DEVICE.type == "cuda":
            torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
        self.t0 = time.perf_counter()
        if self.handle is not None:
            self._thread = threading.Thread(target=self._run, daemon=True); self._thread.start()
        return self

    def __exit__(self, *exc):
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        self.elapsed = time.perf_counter() - self.t0
        if self.handle is not None:
            self._stop.set(); self._thread.join()
        self.peak_mem_mb = torch.cuda.max_memory_allocated() / 2**20 if DEVICE.type == "cuda" else float("nan")

    def summary(self):
        out = dict(seconds=self.elapsed, peak_mem_mb=self.peak_mem_mb,
                   avg_power_w=float("nan"), energy_j=float("nan"), avg_util_pct=float("nan"))
        if len(self.samples) >= 2:
            s = np.array(self.samples)
            out["avg_power_w"]  = float(s[:, 1].mean())
            out["energy_j"]     = float((np.trapezoid if hasattr(np, "trapezoid") else np.trapz)(s[:, 1], s[:, 0]))
            out["avg_util_pct"] = float(s[:, 2].mean())
        elif len(self.samples) == 1:
            out["avg_power_w"] = self.samples[0][1]; out["energy_j"] = self.samples[0][1] * self.elapsed
        return out

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * y.size(0)
        correct  += (out.argmax(1) == y).sum().item()
        total    += y.size(0)
    return loss_sum / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, criterion=None):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        out = model(x)
        if criterion is not None:
            loss_sum += criterion(out, y).item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total   += y.size(0)
    return (loss_sum / total if criterion else float("nan")), 100.0 * correct / total


def fit(model, name, epochs=None):
    epochs = epochs or CFG["EPOCHS"]
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=CFG["LR"], momentum=CFG["MOMENTUM"],
                                weight_decay=CFG["WEIGHT_DECAY"], nesterov=True)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    hist = dict(epoch=[], train_loss=[], train_acc=[], test_loss=[], test_acc=[], epoch_time=[])
    best_acc, best_state = 0.0, None

    with GPUMonitor() as mon:
        for ep in range(1, epochs + 1):
            if DEVICE.type == "cuda": torch.cuda.synchronize()
            t0 = time.perf_counter()
            tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
            if DEVICE.type == "cuda": torch.cuda.synchronize()
            ep_time = time.perf_counter() - t0
            te_loss, te_acc = evaluate(model, test_loader, criterion)
            scheduler.step()
            for k, v in zip(hist, (ep, tr_loss, tr_acc, te_loss, te_acc, ep_time)):
                hist[k].append(v)
            if te_acc > best_acc:
                best_acc, best_state = te_acc, copy.deepcopy(model.state_dict())
            print(f"[{name}] epoch {ep:3d}/{epochs} | train loss {tr_loss:.3f} acc {tr_acc:5.2f}% "
                  f"| test loss {te_loss:.3f} acc {te_acc:5.2f}% | {ep_time:5.1f}s")

    model.load_state_dict(best_state)
    os.makedirs("checkpoints", exist_ok=True)
    torch.save(best_state, f"checkpoints/{name}.pt")
    train_stats = mon.summary()
    train_stats.update(best_test_acc=best_acc, final_test_acc=hist["test_acc"][-1],
                       train_time_s=sum(hist["epoch_time"]),
                       sec_per_epoch=float(np.mean(hist["epoch_time"])))
    return model, pd.DataFrame(hist), train_stats


@torch.no_grad()
def benchmark_inference(model, loader, n_latency=200):
    model.eval().to(DEVICE)
    # warm-up
    x, _ = next(iter(loader))
    for _ in range(5): model(x.to(DEVICE))
    # throughput over the full test set (measured with power monitor)
    with GPUMonitor() as mon:
        n = 0
        for x, _ in loader:
            model(x.to(DEVICE, non_blocking=True)); n += x.size(0)
    s = mon.summary()
    # batch-1 latency
    x1 = x[:1].to(DEVICE)
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_latency): model(x1)
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    lat_ms = (time.perf_counter() - t0) / n_latency * 1000
    return dict(throughput_img_s=n / s["seconds"], latency_ms_bs1=lat_ms,
                infer_peak_mem_mb=s["peak_mem_mb"], infer_avg_power_w=s["avg_power_w"],
                infer_energy_j=s["energy_j"],
                infer_energy_mj_per_img=s["energy_j"] / n * 1000 if not math.isnan(s["energy_j"]) else float("nan"))


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

## 5. ANN Baseline (CNN)

To keep the comparison fair, the CNN and the SNN share **the same VGG-style topology**: the same convolutions, channel widths, pooling, and classifier. The only differences are the activation (ReLU vs. LIF spiking neuron) and the normalization (BatchNorm vs. TEBN).

```
[64, 64, M, 128, 128, M, 256, 256, M] -> FC 4096->256 -> FC 256->10
```

Every convolution is a 3×3 conv → BN → activation block. `M` = 2×2 max-pool.

In [ ]:
VGG_CFG = [64, 64, "M", 128, 128, "M", 256, 256, "M"]

class CNN(nn.Module):
    def __init__(self, cfg=VGG_CFG, num_classes=10):
        super().__init__()
        layers, c_in = [], 3
        for v in cfg:
            if v == "M":
                layers.append(nn.MaxPool2d(2))
            else:
                layers += [nn.Conv2d(c_in, v, 3, padding=1, bias=False), nn.BatchNorm2d(v), nn.ReLU(inplace=True)]
                c_in = v
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(c_in * 4 * 4, 256), nn.ReLU(inplace=True),
                                        nn.Linear(256, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))

print(f"CNN parameters: {count_params(CNN()):,}")

## 6. SNN: LIF Neurons, Surrogate Gradients, and TEBN

### 6.1 Leaky Integrate-and-Fire (LIF) neuron
At each timestep $t$ the membrane potential $u$ leaks, integrates the input current $x$, and fires a spike when it crosses the threshold $V_{th}$. It then hard-resets:

$$u^{t} = \tau\,u^{t-1}(1 - s^{t-1}) + x^{t}, \qquad s^{t} = \Theta(u^{t} - V_{th})$$

### 6.2 Surrogate-gradient backpropagation
The Heaviside step $\Theta$ has zero gradient almost everywhere. In the backward pass we swap in a triangle surrogate, as in the official TEBN code:

$$\frac{\partial s}{\partial u} \approx \frac{1}{\gamma^2}\max(0,\ \gamma - |u - V_{th}|)$$

Training then uses backpropagation through time (BPTT) across the $T$ timesteps.

### 6.3 Temporal Effective Batch Normalization (TEBN)
Standard BN in SNNs either ignores time or normalizes each timestep separately. Because spikes are produced by a temporal process, either choice skews the distribution of membrane inputs. TEBN does two things:

1. **Shared temporal statistics:** it computes the BN mean and variance jointly over the batch **and all timesteps**, i.e. over $(B\cdot T, H, W)$. The affine $\gamma, \beta$ are shared too.
2. **Per-timestep rescaling:** it multiplies the normalized output at timestep $t$ by a learnable scalar $p_t$, which lets the network weight timesteps differently:

$$\text{TEBN}(x^{t}) = p_t \cdot \Big(\gamma\,\frac{x^{t} - \mu_{B,T}}{\sqrt{\sigma^2_{B,T} + \epsilon}} + \beta\Big)$$

The paper shows that this smooths the temporal distribution of membrane potentials and improves accuracy at low timesteps.

### 6.4 Input encoding & readout
- **Direct encoding:** the normalized image is repeated across $T$ timesteps, and the first conv + TEBN + LIF layer acts as the spike encoder.
- **Readout:** the final linear layer's outputs are averaged over $T$ and trained with cross-entropy.

Tensor shape convention inside the SNN: `[B, T, C, H, W]`.

In [ ]:
class SpikeFn(torch.autograd.Function):
    # Heaviside forward, triangle surrogate gradient backward.
    @staticmethod
    def forward(ctx, x, gamma):
        ctx.save_for_backward(x); ctx.gamma = gamma
        return (x >= 0).to(x.dtype)

    @staticmethod
    def backward(ctx, grad_out):
        (x,), g = ctx.saved_tensors, ctx.gamma
        sg = (1.0 / g**2) * (g - x.abs()).clamp(min=0)
        return grad_out * sg, None


class LIF(nn.Module):
    # Multi-step LIF neuron. Input/Output: [B, T, ...]. Records firing rate for analysis.
    def __init__(self, tau=CFG["TAU"], v_th=CFG["V_TH"], gamma=CFG["SG_GAMMA"]):
        super().__init__()
        self.tau, self.v_th, self.gamma = tau, v_th, gamma
        self.record, self.rates = False, []

    def forward(self, x):
        u = torch.zeros_like(x[:, 0])
        spikes = []
        for t in range(x.shape[1]):
            u = self.tau * u + x[:, t]
            s = SpikeFn.apply(u - self.v_th, self.gamma)
            u = u * (1.0 - s)                     # hard reset
            spikes.append(s)
        out = torch.stack(spikes, dim=1)
        if self.record:
            self.rates.append(out.detach().mean(dim=tuple(d for d in range(out.dim()) if d != 1)).cpu())  # per-timestep rate
        return out


class SeqToANN(nn.Module):
    # Applies a stateless layer to every timestep by folding T into the batch: [B,T,...] -> [B*T,...] -> [B,T,...]
    def __init__(self, *mods):
        super().__init__(); self.m = nn.Sequential(*mods)
    def forward(self, x):
        B, Tt = x.shape[:2]
        y = self.m(x.flatten(0, 1))
        return y.view(B, Tt, *y.shape[1:])


class TEBN(nn.Module):
    # Temporal Effective BN (Duan et al., NeurIPS 2022):
    # BN statistics shared over batch and time + learnable per-timestep scale p_t.
    def __init__(self, channels, T=CFG["TIMESTEPS"], eps=1e-5, momentum=0.1):
        super().__init__()
        self.bn = nn.BatchNorm2d(channels, eps=eps, momentum=momentum)
        self.p = nn.Parameter(torch.ones(1, T, 1, 1, 1))
    def forward(self, x):                           # x: [B, T, C, H, W]
        B, Tt = x.shape[:2]
        y = self.bn(x.flatten(0, 1)).view_as(x)     # stats over (B*T, H, W)
        return y * self.p[:, :Tt]


class PlainBN(nn.Module):
    # Baseline used in the ablation: ordinary BN applied per timestep (no time-scaling).
    def __init__(self, channels, **_):
        super().__init__(); self.bn = nn.BatchNorm2d(channels)
    def forward(self, x):
        return self.bn(x.flatten(0, 1)).view_as(x)


class SNN(nn.Module):
    def __init__(self, cfg=VGG_CFG, num_classes=10, T=CFG["TIMESTEPS"], norm=TEBN):
        super().__init__()
        self.T = T
        layers, c_in = [], 3
        for v in cfg:
            if v == "M":
                layers.append(SeqToANN(nn.MaxPool2d(2)))
            else:
                layers += [SeqToANN(nn.Conv2d(c_in, v, 3, padding=1, bias=False)), norm(v, T=T), LIF()]
                c_in = v
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(SeqToANN(nn.Flatten(), nn.Linear(c_in * 4 * 4, 256)), LIF(),
                                        SeqToANN(nn.Linear(256, num_classes)))

    def forward(self, x):                                   # x: [B, C, H, W]
        x = x.unsqueeze(1).repeat(1, self.T, 1, 1, 1)       # direct encoding -> [B, T, C, H, W]
        out = self.classifier(self.features(x))             # [B, T, classes]
        return out.mean(1)                                  # rate-decoded logits

    def lif_layers(self):
        return [m for m in self.modules() if isinstance(m, LIF)]

print(f"SNN parameters: {count_params(SNN()):,}  (same topology + {CFG['TIMESTEPS']} p_t scalars per TEBN layer)")

In [ ]:
# Sanity checks: shapes, binary spikes, and a non-zero surrogate gradient
_m = SNN().to(DEVICE)
_x = torch.randn(8, 3, 32, 32, device=DEVICE, requires_grad=True)
_o = _m(_x); _o.sum().backward()
print("output:", tuple(_o.shape), "| input grad norm:", _x.grad.norm().item())
_s = LIF()(torch.randn(2, 4, 10) * 2)
assert set(_s.unique().tolist()) <= {0.0, 1.0}, "LIF output must be binary"
del _m, _x, _o, _s

## 7. Training & Evaluation: ANN Baseline

Both models use the same optimizer: SGD with Nesterov momentum 0.9, lr 0.1, weight decay 5e-4, a cosine schedule, and batch size 128. The best checkpoint by test accuracy is kept.

In [ ]:
set_seed(CFG["SEED"])
ann, ann_hist, ann_train = fit(CNN(), "ANN_CNN")
ann_infer = benchmark_inference(ann, test_loader)
print("\nTraining:", {k: round(v, 3) for k, v in ann_train.items()})
print("Inference:", {k: round(v, 3) for k, v in ann_infer.items()})

## 8. Training & Evaluation: SNN (LIF + Surrogate Gradient + TEBN)

The SNN unrolls $T$ timesteps, so each forward/backward pass costs roughly $T\times$ the CNN's work on a GPU. Expect longer epochs, which is the hardware challenge noted in the proposal.

In [ ]:
set_seed(CFG["SEED"])
snn, snn_hist, snn_train = fit(SNN(norm=TEBN), "SNN_TEBN")
snn_infer = benchmark_inference(snn, test_loader)
print("\nTraining:", {k: round(v, 3) for k, v in snn_train.items()})
print("Inference:", {k: round(v, 3) for k, v in snn_infer.items()})

In [ ]:
# Learned TEBN per-timestep scales p_t
p = torch.stack([m.p.detach().flatten().cpu() for m in snn.modules() if isinstance(m, TEBN)])
plt.figure(figsize=(6, 3.5))
plt.imshow(p.numpy(), aspect="auto", cmap="viridis"); plt.colorbar(label="p_t")
plt.xlabel("timestep t"); plt.ylabel("TEBN layer"); plt.title("Learned TEBN time-scales")
plt.xticks(range(p.shape[1])); plt.tight_layout(); plt.show()

## 9. Spike-Activity Analysis

We record the **firing rate** of every LIF layer on the test set, i.e. the fraction of neurons that spike, per layer and per timestep. Sparse activity is the source of an SNN's potential energy advantage, because on neuromorphic hardware a neuron that doesn't fire triggers no computation.

In [ ]:
@torch.no_grad()
def spike_rates(model, loader, max_batches=None):
    lifs = model.lif_layers()
    for l in lifs: l.record, l.rates = True, []
    model.eval()
    for i, (x, _) in enumerate(loader):
        model(x.to(DEVICE))
        if max_batches and i + 1 >= max_batches: break
    per_t = torch.stack([torch.stack(l.rates).mean(0) for l in lifs])   # [layers, T]
    for l in lifs: l.record, l.rates = False, []
    return per_t

rates = spike_rates(snn, test_loader)
layer_names = [f"LIF{i+1}" for i in range(rates.shape[0])]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(layer_names, rates.mean(1).numpy() * 100)
ax[0].set_ylabel("firing rate (%)"); ax[0].set_title("Average firing rate per layer")
for t in range(rates.shape[1]):
    ax[1].plot(layer_names, rates[:, t].numpy() * 100, marker="o", label=f"t={t}")
ax[1].set_ylabel("firing rate (%)"); ax[1].set_title("Firing rate per timestep"); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"Network-wide average firing rate: {rates.mean().item()*100:.2f}%")

In [ ]:
# Raster plot: spikes of 100 random neurons in the first LIF layer, for one test image
lif1 = snn.lif_layers()[0]
cache = {}
h = lif1.register_forward_hook(lambda m, i, o: cache.setdefault("s", o.detach().cpu()))
with torch.no_grad(): snn.eval()(test_set[0][0].unsqueeze(0).to(DEVICE))
h.remove()
s = cache["s"][0].flatten(1)                       # [T, neurons]
idx = torch.randperm(s.shape[1])[:100]
plt.figure(figsize=(8, 3.5))
for t in range(s.shape[0]):
    plt.scatter([t] * len(n), torch.nonzero(s[t, idx] > 0).flatten(), marker="|", s=200, c="k")
plt.xlabel("timestep"); plt.ylabel("neuron index"); plt.title("Spike raster, LIF layer 1 (100 neurons)")
plt.xticks(range(s.shape[0])); plt.tight_layout(); plt.show()

## 10. Theoretical Energy Estimate (MACs vs. SOPs)

A GPU is a dense, synchronous accelerator. It processes zeros (non-spikes) just as it processes ones, so **measured GPU power does not reflect what an SNN would cost on neuromorphic hardware** (Intel Loihi, IBM TrueNorth, SpiNNaker). The standard estimate in the SNN literature, which the TEBN paper's line of work also uses, counts operations:

- **ANN:** every layer performs dense multiply–accumulates. $E_{ANN} = \text{MACs} \times E_{MAC}$
- **SNN:** the first layer receives real-valued input (MACs at every timestep). Later layers receive binary spikes, so each synaptic operation is just an accumulate, triggered only when a spike arrives:
  $E_{SNN} = \text{MACs}_{1}\cdot T\cdot E_{MAC} + \sum_{l>1} r_{l}\cdot T\cdot \text{MACs}_{l}\cdot E_{AC}$

where $r_l$ is the measured firing rate of the input to layer $l$, $E_{MAC}=4.6$ pJ and $E_{AC}=0.9$ pJ (45 nm CMOS).

The code below hooks every conv/linear layer, counts its MACs, and measures the fraction of non-zero (spiking) inputs. It treats a layer as spike-driven only if its input is strictly binary.

In [ ]:
@torch.no_grad()
def op_profile(model, loader, max_batches=10):
    # Returns a per-layer table of MACs/sample, input activity and binary-ness.
    stats, hooks = {}, []
    def mk(name, mod):
        def hook(m, inp, out):
            x = inp[0]
            if isinstance(m, nn.Conv2d):
                macs = out[0].numel() * (m.in_channels // m.groups) * m.kernel_size[0] * m.kernel_size[1]
            else:
                macs = m.in_features * m.out_features
            d = stats.setdefault(name, dict(macs=macs, nz=0.0, n=0, binary=True))
            d["nz"] += (x != 0).float().mean().item(); d["n"] += 1
            d["binary"] &= bool(((x == 0) | (x == 1)).all())
        return hook
    for name, m in model.named_modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            hooks.append(m.register_forward_hook(mk(name, m)))
    model.eval()
    for i, (x, _) in enumerate(loader):
        model(x.to(DEVICE))
        if i + 1 >= max_batches: break
    for h in hooks: h.remove()
    return pd.DataFrame([dict(layer=k, macs=v["macs"], input_rate=v["nz"] / v["n"], spike_input=v["binary"])
                         for k, v in stats.items()])

E_MAC, E_AC, TT = CFG["E_MAC_PJ"] * 1e-12, CFG["E_AC_PJ"] * 1e-12, CFG["TIMESTEPS"]

ann_ops = op_profile(ann, test_loader)
ann_macs = ann_ops["macs"].sum()
ann_energy = ann_macs * E_MAC

snn_ops = op_profile(snn, test_loader)
# For the SNN, hooks see the [B*T] folded batch, so per-sample MACs are per-timestep; multiply by T.
snn_ops["MAC_ops"] = np.where(~snn_ops.spike_input, snn_ops.macs * TT, 0)
snn_ops["SOP_ops"] = np.where(snn_ops.spike_input, snn_ops.macs * TT * snn_ops.input_rate, 0)
snn_energy = (snn_ops.MAC_ops * E_MAC + snn_ops.SOP_ops * E_AC).sum()

display(snn_ops.assign(macs=lambda d: d.macs.map("{:,.0f}".format),
                       input_rate=lambda d: d.input_rate.map("{:.3f}".format)))
print(f"ANN: {ann_macs/1e6:,.1f} M MACs/img -> {ann_energy*1e6:.2f} uJ/img")
print(f"SNN: {snn_ops.MAC_ops.sum()/1e6:,.1f} M MACs + {snn_ops.SOP_ops.sum()/1e6:,.1f} M SOPs /img "
      f"-> {snn_energy*1e6:.2f} uJ/img")
print(f"Estimated neuromorphic energy ratio (ANN / SNN): {ann_energy / snn_energy:.2f}x")

## 11. ANN vs. SNN Comparison

In [ ]:
def row(name, model, tr, inf, theo_uj):
    return {
        "Model": name,
        "Params": f"{count_params(model):,}",
        "Best test acc (%)": round(tr["best_test_acc"], 2),
        "Train time (min)": round(tr["train_time_s"] / 60, 2),
        "Sec / epoch": round(tr["sec_per_epoch"], 1),
        "Train peak GPU mem (MB)": round(tr["peak_mem_mb"], 0),
        "Train avg GPU util (%)": round(tr["avg_util_pct"], 1),
        "Train avg power (W)": round(tr["avg_power_w"], 1),
        "Train energy (kJ)": round(tr["energy_j"] / 1000, 2),
        "Inference throughput (img/s)": round(inf["throughput_img_s"], 0),
        "Latency bs=1 (ms)": round(inf["latency_ms_bs1"], 2),
        "Inference peak mem (MB)": round(inf["infer_peak_mem_mb"], 0),
        "Inference avg power (W)": round(inf["infer_avg_power_w"], 1),
        "GPU energy / img (mJ)": round(inf["infer_energy_mj_per_img"], 3),
        "Theoretical energy / img (uJ)": round(theo_uj, 2),
    }

results = pd.DataFrame([
    row("ANN (CNN)", ann, ann_train, ann_infer, ann_energy * 1e6),
    row(f"SNN (TEBN, T={TT})", snn, snn_train, snn_infer, snn_energy * 1e6),
]).set_index("Model").T
os.makedirs("results", exist_ok=True)
results.to_csv("results/ann_vs_snn.csv")
results

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for h, n in ((ann_hist, "ANN"), (snn_hist, "SNN-TEBN")):
    ax[0].plot(h.epoch, h.test_acc, label=n); ax[1].plot(h.epoch, h.train_loss, label=n)
ax[0].set_title("Test accuracy"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("%"); ax[0].legend()
ax[1].set_title("Training loss"); ax[1].set_xlabel("epoch"); ax[1].legend()

labels = ["Train time\n(min)", "Throughput\n(img/s ÷100)", "GPU energy\n(mJ/img)", "Theoretical\n(uJ/img)"]
a_vals = [ann_train["train_time_s"]/60, ann_infer["throughput_img_s"]/100, ann_infer["infer_energy_mj_per_img"], ann_energy*1e6]
s_vals = [snn_train["train_time_s"]/60, snn_infer["throughput_img_s"]/100, snn_infer["infer_energy_mj_per_img"], snn_energy*1e6]
xx = np.arange(len(labels)); w = 0.38
ax[2].bar(xx - w/2, np.nan_to_num(a_vals), w, label="ANN"); ax[2].bar(xx + w/2, np.nan_to_num(s_vals), w, label="SNN-TEBN")
ax[2].set_xticks(xx); ax[2].set_xticklabels(labels); ax[2].set_title("Cost comparison (lower is better, except throughput)")
ax[2].legend(); plt.tight_layout(); plt.savefig("results/ann_vs_snn.png", dpi=150); plt.show()

## 12. Optional Experiment: TEBN vs. Standard BN

This trains the same SNN with ordinary BatchNorm (no temporal scaling) to isolate the contribution of TEBN. Set `CFG["RUN_ABLATION"] = True` to run it; it doubles SNN training time. You can also sweep `TIMESTEPS` (e.g. 2, 4, 6) to see the accuracy/energy trade-off.

In [ ]:
if CFG["RUN_ABLATION"]:
    set_seed(CFG["SEED"])
    snn_bn, snn_bn_hist, snn_bn_train = fit(SNN(norm=PlainBN), "SNN_PlainBN")
    bn_rates = spike_rates(snn_bn, test_loader)
    abl = pd.DataFrame({
        "SNN + BN":   [snn_bn_train["best_test_acc"], bn_rates.mean().item() * 100],
        "SNN + TEBN": [snn_train["best_test_acc"],    rates.mean().item() * 100],
    }, index=["Best test acc (%)", "Avg firing rate (%)"])
    plt.plot(snn_bn_hist.epoch, snn_bn_hist.test_acc, label="SNN + BN")
    plt.plot(snn_hist.epoch, snn_hist.test_acc, label="SNN + TEBN")
    plt.xlabel("epoch"); plt.ylabel("test acc (%)"); plt.legend(); plt.title("TEBN ablation"); plt.show()
    display(abl.round(2))
else:
    print("Ablation skipped (set CFG['RUN_ABLATION'] = True to run).")

## 13. Expected Challenges

- **Hardware limitations.** The goal is to train both models on a **GTX 1080 Ti**. The SNN unrolls $T$ timesteps and stores activations for BPTT, so it needs roughly $T\times$ more compute and memory per step than the CNN. If this is a problem, lower `BATCH_SIZE` or `TIMESTEPS`, or train on a cloud platform such as **Kaggle** or **Google Colab**. The notebook runs unchanged there.
- **GPU vs. neuromorphic energy.** GPUs don't exploit spike sparsity, so measured GPU power will likely *favor the ANN*. That is why Section 10 adds the operation-count estimate, which approximates the cost on neuromorphic hardware.
- **Dataset.** CIFAR-10 is balanced and well established, so no major data issues are expected. However, data augmentation is required to approach the paper's reported results. The paper reports 94.70% for ResNet-19 at T=4 after 300 epochs with stronger augmentation. This simplified VGG at 30 epochs will land below that.

## 14. Team Roles

We will share work and implementation so both of us understand both sides, but our roles specialize as follows.

| Ethan | Marcus |
|---|---|
| Implement SNN and reproduce TEBN method (Sec. 6, 8) | Prepare and preprocess CIFAR-10 dataset (Sec. 2) |
| Perform spike-activity and SNN experiments (Sec. 9, 10, 12) | Implement and train CNN model (Sec. 5, 7) |
| Record metrics and compare with CNN (Sec. 11) | Record metrics and compare with SNN (Sec. 11) |

## 15. Conclusions

*Fill in after running the experiments.* Questions to answer from the results above:

1. **Accuracy:** How close does the SNN (TEBN) come to the CNN baseline at the same topology?
2. **Training cost:** How much longer and more memory-hungry is SNN training on a GPU?
3. **Inference on GPU:** Throughput, latency, and measured energy per image for each model.
4. **Neuromorphic potential:** Given the measured firing rates, what is the estimated energy ratio, and would it justify SNNs for data-center-scale deployment?
5. **TEBN contribution:** (if the ablation was run) How much does TEBN improve over plain BN?